In [6]:

import re

# Variables used in this program.
# Lowercase a and b are treated as constants, not variables.
VARIABLES = {"x", "y", "z", "u", "v", "w"}

# Tokenize identifiers and symbols.
TOKEN_PATTERN = r"[A-Za-z_][A-Za-z0-9_]*|[(){};,]"


def tokenize(text):
    tokens = re.findall(TOKEN_PATTERN, text)
    if "".join(tokens) != re.sub(r"\s+", "", text):
        raise ValueError("Invalid expression syntax.")
    return tokens


def parse_term(tokens, pos):
    if pos >= len(tokens):
        raise ValueError("Unexpected end of expression.")

    name = tokens[pos]
    pos += 1

    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError("Expected a term.")

    # Parse a predicate or nested function.
    if pos < len(tokens) and tokens[pos] == "(":
        pos += 1
        args = []

        if pos < len(tokens) and tokens[pos] != ")":
            while True:
                arg, pos = parse_term(tokens, pos)
                args.append(arg)

                if pos < len(tokens) and tokens[pos] == ",":
                    pos += 1
                    continue
                break

        if pos >= len(tokens) or tokens[pos] != ")":
            raise ValueError("Missing closing parenthesis.")

        pos += 1
        return [name] + args, pos

    return name, pos


def parse_input(text):
    tokens = tokenize(text)

    if not tokens:
        raise ValueError("Expression cannot be empty.")

    # Parse a set of predicates: {p(...); p(...)}
    if tokens[0] == "{":
        pos = 1
        expressions = []

        while pos < len(tokens) and tokens[pos] != "}":
            term, pos = parse_term(tokens, pos)
            expressions.append(term)

            if pos < len(tokens) and tokens[pos] == ";":
                pos += 1
            elif pos < len(tokens) and tokens[pos] != "}":
                raise ValueError("Expected ';' or '}'.")

        if pos >= len(tokens) or tokens[pos] != "}":
            raise ValueError("Missing closing brace.")

        pos += 1
        if pos != len(tokens):
            raise ValueError("Unexpected text after closing brace.")

        return ["AND"] + expressions

    expression, pos = parse_term(tokens, 0)

    if pos != len(tokens):
        raise ValueError("Unexpected text after expression.")

    return expression


def is_variable(term):
    return isinstance(term, str) and term in VARIABLES


def apply_substitution(term, subst):
    if is_variable(term) and term in subst:
        return apply_substitution(subst[term], subst)

    if isinstance(term, list):
        return [apply_substitution(item, subst) for item in term]

    return term


def occurs_check(var, term, subst):
    term = apply_substitution(term, subst)

    if var == term:
        return True

    if isinstance(term, list):
        return any(occurs_check(var, item, subst) for item in term)

    return False


def unify_terms(left, right, subst):
    left = apply_substitution(left, subst)
    right = apply_substitution(right, subst)

    if left == right:
        return True

    if is_variable(left):
        if occurs_check(left, right, subst):
            return False
        subst[left] = right
        return True

    if is_variable(right):
        if occurs_check(right, left, subst):
            return False
        subst[right] = left
        return True

    if isinstance(left, list) and isinstance(right, list):
        if len(left) != len(right):
            return False

        for a, b in zip(left, right):
            if not unify_terms(a, b, subst):
                return False

        return True

    return False


def unify(expr1, expr2):
    subst = {}

    if unify_terms(expr1, expr2, subst):
        # Resolve all substitutions.
        for var in list(subst):
            subst[var] = apply_substitution(subst[var], subst)

        return subst

    return None


def main():
    print("=== First-Order Logic Unification System ===")
    print("Variables = x, y, z, u, v, w")
    print("Constants can be uppercase or lowercase.")
    print("Example: Eats(x, Apple)")
    print("Nested example: p(b, x, f(g(z)))")

    test_no = 1

    while True:
        try:
            raw1 = input("\nEnter Expression 1: ")
            raw2 = input("Enter Expression 2: ")

            expr1 = parse_input(raw1)
            expr2 = parse_input(raw2)

            # Display nested sets in the original format.
            if raw1.strip().startswith("{"):
                print("\nParsed Expression 1:", raw1.strip())
            else:
                print("\nParsed Expression 1:", expr1)

            if raw2.strip().startswith("{"):
                print("Parsed Expression 2:", raw2.strip())
            else:
                print("Parsed Expression 2:", expr2)

            result = unify(expr1, expr2)

            print("\n--- RESULT ---")

            if result is None:
                print("Unification Failed!")
                print("Reason: Mismatched terms or predicates.")

            elif not result:
                print("Unification Successful!")
                print("Expressions are identical. No substitutions needed.")

            else:
                print("Unification Successful!")
                print("Substitutions (MGU):")

                for var, value in result.items():
                    print(f"  {var} / {value}")

            print("\n" + "=" * 45)
            test_no += 1

        except (ValueError, RecursionError) as error:
            print("\nInvalid expression:", error)
            print("=" * 45)

        choice = input(
            "\nDo you want to test another pair? (yes/no): "
        ).strip().lower()

        if choice != "yes":
            print("Program terminated.")
            break

        print()


if __name__ == "__main__":
    main()


=== First-Order Logic Unification System ===
Variables = x, y, z, u, v, w
Constants can be uppercase or lowercase.
Example: Eats(x, Apple)
Nested example: p(b, x, f(g(z)))

Enter Expression 1: Eats(x, Apple)
Enter Expression 2: Eats(John, Apple)

Parsed Expression 1: ['Eats', 'x', 'Apple']
Parsed Expression 2: ['Eats', 'John', 'Apple']

--- RESULT ---
Unification Successful!
Substitutions (MGU):
  x / John


Do you want to test another pair? (yes/no): yes


Enter Expression 1: Likes(John, Apple)
Enter Expression 2: Likes(Mary, Apple)

Parsed Expression 1: ['Likes', 'John', 'Apple']
Parsed Expression 2: ['Likes', 'Mary', 'Apple']

--- RESULT ---
Unification Failed!
Reason: Mismatched terms or predicates.


Do you want to test another pair? (yes/no): yes


Enter Expression 1: {p(b, x, f(g(z))); p(z, f(y), f(y))}
Enter Expression 2: {p(b, x, f(g(b))); p(b, f(y), f(y))}

Parsed Expression 1: {p(b, x, f(g(z))); p(z, f(y), f(y))}
Parsed Expression 2: {p(b, x, f(g(b))); p(b, f(y), f(y))}

---